# 01 — Data Collection

Bu notebook'ta iki kaynaktan veri çekiyoruz:
- **yfinance**: Apple (AAPL) yıllık finansal tabloları (gelir, FCF, net gelir)
- **FRED (Federal Reserve)**: Makroekonomik göstergeler (Fed faizi, CPI, GDP büyümesi, USD endeksi)

Çıktı: `data/processed/apple_macro_combined.csv`

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_fetcher import (
    fetch_apple_financials,
    fetch_all_macro,
    build_combined_dataset,
    fetch_apple_beta,
    fetch_shares_outstanding,
    FRED_SERIES
)

print('Libraries loaded successfully.')

TypeError: deprecate_kwarg() missing 1 required positional argument: 'new_arg_name'

## 1. Apple Finansal Verileri

In [ ]:
apple_df = fetch_apple_financials()
print(f'Apple finansal veri: {len(apple_df)} yıl')
print(f'Kolonlar: {list(apple_df.columns)}')
apple_df

In [ ]:
# Apple FCF ve Gelir zaman serisi
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'Total Revenue' in apple_df.columns:
    axes[0].bar(apple_df.index.year if hasattr(apple_df.index[0], 'year') else apple_df.index,
                apple_df['Total Revenue'] / 1e9, color='#3498db', alpha=0.8)
    axes[0].set_title('Apple Yıllık Gelir ($B)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Milyar USD')
    axes[0].grid(axis='y', alpha=0.3)

if 'Free Cash Flow' in apple_df.columns:
    axes[1].bar(apple_df.index.year if hasattr(apple_df.index[0], 'year') else apple_df.index,
                apple_df['Free Cash Flow'] / 1e9, color='#2ecc71', alpha=0.8)
    axes[1].set_title('Apple Yıllık Serbest Nakit Akışı ($B)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Milyar USD')
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/apple_financials_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 2. FRED Makroekonomik Göstergeler

In [ ]:
print('FRED series çekiliyor...')
macro_df = fetch_all_macro()
print(f'Makro veri: {len(macro_df)} yıl, {len(macro_df.columns)} gösterge')
macro_df.tail(10)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

series_labels = {
    'fed_rate': ('Fed Funds Rate (%)', '#e74c3c'),
    'cpi': ('CPI (Enflasyon Endeksi)', '#f39c12'),
    'gdp_growth': ('GDP Büyümesi (%)', '#27ae60'),
    'usd_index': ('USD Endeksi', '#8e44ad'),
}

for i, (col, (label, color)) in enumerate(series_labels.items()):
    if col in macro_df.columns:
        axes[i].plot(macro_df.index, macro_df[col], color=color, linewidth=2, marker='o', markersize=4)
        axes[i].set_title(label, fontsize=11, fontweight='bold')
        axes[i].grid(alpha=0.3)
        axes[i].set_xlabel('Yıl')

plt.suptitle('Makroekonomik Göstergeler (Yıllık Ortalama)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/macro_indicators_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Birleşik Dataset Oluşturma

In [ ]:
combined_df = build_combined_dataset(save_path='../data/processed/apple_macro_combined.csv')
print(f'\nBirleşik dataset: {combined_df.shape[0]} yıl, {combined_df.shape[1]} değişken')
combined_df

## 4. Apple Beta & Hisse Bilgisi

In [ ]:
beta = fetch_apple_beta()
shares = fetch_shares_outstanding()

print(f'Apple Beta (5Y Monthly): {beta:.2f}')
print(f'Dolaşımdaki Hisse Sayısı: {shares/1e9:.2f}B')
print('\nBu değerler Notebook 03 DCF modeli için kullanılacak.')

## Özet

| Veri | Kaynak | Durum |
|------|--------|-------|
| Apple Finansallar | yfinance | ✅ |
| Fed Faizi | FRED: FEDFUNDS | ✅ |
| CPI Enflasyon | FRED: CPIAUCSL | ✅ |
| GDP Büyümesi | FRED: A191RL1Q225SBEA | ✅ |
| USD Endeksi | FRED: DTWEXBGS | ✅ |
| 10Y Treasury | FRED: DGS10 | ✅ |

Birleşik dataset `data/processed/apple_macro_combined.csv` olarak kaydedildi. Sonraki adım: **02_macro_correlation_analysis.ipynb**